In [15]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
from pathlib import Path

In [16]:
file_path = Path('../data/processed/ipl_cleaned.csv')
df = pd.read_csv(file_path,low_memory=False)

In [17]:
# 2. Engineer the 'bowling_team' column
df['bowling_team'] = np.where(
    df['batting_team'] == df['team1'], 
    df['team2'], 
    df['team1']
)

# 3. Verify the logic worked perfectly
df[['team1', 'team2', 'batting_team', 'bowling_team']].head()

,team1,team2,batting_team,bowling_team
0,Royal Challengers Bengaluru,Kolkata Knight Riders,Kolkata Knight Riders,Royal Challengers Bengaluru
1,Royal Challengers Bengaluru,Kolkata Knight Riders,Kolkata Knight Riders,Royal Challengers Bengaluru
2,Royal Challengers Bengaluru,Kolkata Knight Riders,Kolkata Knight Riders,Royal Challengers Bengaluru
3,Royal Challengers Bengaluru,Kolkata Knight Riders,Kolkata Knight Riders,Royal Challengers Bengaluru
4,Royal Challengers Bengaluru,Kolkata Knight Riders,Kolkata Knight Riders,Royal Challengers Bengaluru


In [18]:
# 1. Engineer the 'is_wicket' column
df['is_wicket'] = np.where(df['wicket_kind'] == 'No Wicket', 0, 1)

# 2. Verify it worked by checking the value counts
# You should see a massive number of 0s and a small number of 1s
print("Total Deliveries vs. Total Wickets:")
print(df['is_wicket'].value_counts())

Total Deliveries vs. Total Wickets:
is_wicket
0    275267
1     14406
Name: count, dtype: int64


In [19]:
# 1. Engineer the 'is_legal_delivery' column
df['is_legal_delivery'] = np.where(
    (df['extras_wides'] > 0) | (df['extras_noballs'] > 0), 
    0, 
    1
)

# 2. Verify the logic by checking the value counts
print("Legal vs. Illegal Deliveries:")
print(df['is_legal_delivery'].value_counts())

Legal vs. Illegal Deliveries:
is_legal_delivery
1    278871
0     10802
Name: count, dtype: int64


In [20]:
# Engineer the 'is_dot_ball' column
df['is_dot_ball'] = np.where(df['runs_total'] == 0, 1, 0)

# Verify the features
print("Dot Balls vs Scoring Deliveries:")
print(df['is_dot_ball'].value_counts())

Dot Balls vs Scoring Deliveries:
is_dot_ball
0    190347
1     99326
Name: count, dtype: int64


In [21]:
# Engineer the 'is_boundary' column using .isin() for a clean check
df['is_boundary'] = np.where(df['runs_batter'].isin([4, 6]), 1, 0)

print("\nBoundaries vs Non-Boundaries:")
print(df['is_boundary'].value_counts())


Boundaries vs Non-Boundaries:
is_boundary
0    240778
1     48895
Name: count, dtype: int64


In [22]:
# 1. Isolate the first innings
first_innings = df[df['innings'] == 1].copy()

# 2. Group by match to count total legal balls and wickets
match_summaries = first_innings.groupby('match_id').agg(
    total_balls=('is_legal_delivery', 'sum'),
    total_wickets=('is_wicket', 'sum')
).reset_index()

# 3. Apply the logic: Less than 115 balls AND not bowled out (less than 10 wickets)
match_summaries['is_rain_reduced'] = (
    (match_summaries['total_balls'] < 115) & 
    (match_summaries['total_wickets'] < 10)
).astype(int)

# 4. Merge this flag back into the main DataFrame
df = df.merge(match_summaries[['match_id', 'is_rain_reduced']], on='match_id', how='left')

# 5. Fill any potential nulls with 0 and ensure integer type
df['is_rain_reduced'] = df['is_rain_reduced'].fillna(0).astype(int)

print(f"Total Rain-Reduced Matches Flagged: {df['is_rain_reduced'].sum()}")

Total Rain-Reduced Matches Flagged: 3389


In [23]:
# 1. Define the exact mathematical conditions for each phase
conditions = [
    (df['over'] <= 5),
    (df['over'] >= 6) & (df['over'] <= 14),
    (df['over'] >= 15)
]

# 2. Define the exact text labels that match those conditions
choices = ['Powerplay', 'Middle Overs', 'Death Overs']

# 3. Create the new column using np.select
# (The 'default' acts as a safety net in case an over > 19 somehow exists)
df['phase'] = np.select(conditions, choices, default='Unknown')

# 4. Verify the distribution of your data across the phases
print("--- Deliveries per Match Phase ---")
print(df['phase'].value_counts())

--- Deliveries per Match Phase ---
phase
Middle Overs    132841
Powerplay        91014
Death Overs      65818
Name: count, dtype: int64


In [24]:
# 1. Calculate the running total of runs for every specific innings
df['current_score'] = df.groupby(['match_id', 'innings'])['runs_total'].cumsum()

# 2. Calculate the running total of wickets for every specific innings
df['current_wickets'] = df.groupby(['match_id', 'innings'])['is_wicket'].cumsum()

# 3. Verify the logic by looking at the first 10 balls of the very first match
columns_to_view = ['match_id', 'innings', 'over', 'ball', 'runs_total', 'current_score', 'is_wicket', 'current_wickets']
print(df[columns_to_view].head(5))

   match_id  innings  over  ball  runs_total  current_score  is_wicket  \
0    335982        1     0     1           1              1          0   
1    335982        1     0     2           0              1          0   
2    335982        1     0     3           1              2          0   
3    335982        1     0     4           0              2          0   
4    335982        1     0     5           0              2          0   

   current_wickets  
0                0  
1                0  
2                0  
3                0  
4                0  


In [25]:
# --- STEP 1: Find the Target Score ---
# Group the 1st innings by match and sum the total runs
first_innings_totals = df[df['innings'] == 1].groupby('match_id')['runs_total'].sum().reset_index()

# The target is always the 1st innings total + 1
first_innings_totals['target'] = first_innings_totals['runs_total'] + 1

# Merge this target score back into our main dataset
df = df.merge(first_innings_totals[['match_id', 'target']], on='match_id', how='left')


# --- STEP 2: Calculate Runs Needed ---
# If it's the 2nd innings, subtract current score from target. Otherwise, leave it blank (NaN).
df['runs_needed'] = np.where(
    df['innings'] == 2, 
    df['target'] - df['current_score'], 
    np.nan
)

# --- STEP 3: Calculate Balls Remaining ---
# First, get a running total of LEGAL deliveries bowled in the innings
df['current_legal_balls'] = df.groupby(['match_id', 'innings'])['is_legal_delivery'].cumsum()

# If it's the 2nd innings, subtract current legal balls from 120. Otherwise, leave blank.
df['balls_remaining'] = np.where(
    df['innings'] == 2, 
    120 - df['current_legal_balls'], 
    np.nan
)

# --- STEP 4: Verify the Magic ---
# Look at the final 5 balls of a tight run chase (e.g., match_id = 335983)
sample_chase = df[(df['match_id'] == 335983) & (df['innings'] == 2)]
columns_to_view = ['over', 'ball', 'runs_total', 'current_score', 'target', 'runs_needed', 'balls_remaining']
print(sample_chase[columns_to_view].tail(5))

     over  ball  runs_total  current_score  target  runs_needed  \
468    19     2           1            200     241         41.0   
469    19     3           0            200     241         41.0   
470    19     4           6            206     241         35.0   
471    19     5           1            207     241         34.0   
472    19     6           0            207     241         34.0   

     balls_remaining  
468              4.0  
469              3.0  
470              2.0  
471              1.0  
472              0.0  


In [26]:
# --- THE SUPER OVER SPLIT ---
# Filter for standard T20 innings (1 and 2)
regular_df = df[df['innings'] <= 2].copy()

# Filter for Super Overs (Innings 3, 4, etc.)
super_overs_df = df[df['innings'] > 2].copy()


# --- THE EXPORT ---
output_dir = Path('../data/processed')

# Define the two separate file paths
regular_file = output_dir / 'ipl_features_regular.csv'
super_over_file = output_dir / 'ipl_features_super_overs.csv'

# Save both DataFrames
regular_df.to_csv(regular_file, index=False)
super_overs_df.to_csv(super_over_file, index=False)

print(f"Standard Match Data saved: {len(regular_df)} rows")
print(f"Super Over Data saved: {len(super_overs_df)} rows")

Standard Match Data saved: 289498 rows
Super Over Data saved: 175 rows
